# 01 - Explorarea Datelor (EDA)

In acest notebook exploram datasetul `popular_10k_mood_labeled.csv`:
- Informatii generale despre dataset
- Distributia mood-urilor
- Statistici descriptive pe feature-uri audio
- Corelatii intre feature-uri
- Distributia feature-urilor pe mood

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import DATASET_PATH, FEATURE_COLUMNS, MOOD_LABELS

## 1. Incarcarea Datelor

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape[0]} randuri x {df.shape[1]} coloane')
print(f'Valori lipsa: {df.isnull().sum().sum()}')
print(f'Coloane: {list(df.columns)}')

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Distributia Mood-urilor

In [ ]:
mood_counts = df['mood'].value_counts().reindex([MOOD_LABELS[i] for i in range(8)])

colors = ['#5B5EA6', '#3D3D6B', '#4D96FF', '#6BCB77', '#FF6B6B', '#FFD93D', '#FF8E53', '#E84393']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
mood_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Distributia Mood-urilor', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Mood')
axes[0].set_ylabel('Numar piese')
axes[0].tick_params(axis='x', rotation=45)

# Add counts on bars
for i, (idx, val) in enumerate(mood_counts.items()):
    axes[0].text(i, val + 50, str(val), ha='center', fontsize=9, fontweight='bold')

# Pie chart
axes[1].pie(mood_counts.values, labels=mood_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
axes[1].set_title('Proportia Mood-urilor', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/mood_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDezechilibru: max={mood_counts.max()} ({mood_counts.idxmax()}), min={mood_counts.min()} ({mood_counts.idxmin()})')
print(f'Raport: {mood_counts.max()/mood_counts.min():.1f}x')

## 3. Feature-uri Audio - Statistici

In [ ]:
print('Statistici descriptive pe feature-uri audio:')
df[FEATURE_COLUMNS].describe().round(3)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLUMNS):
    axes[i].hist(df[feat], bins=40, color=colors[i % len(colors)], edgecolor='white', alpha=0.85)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Frecventa')

plt.suptitle('Distributia Feature-urilor Audio', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Matricea de Corelatie

In [ ]:
corr = df[FEATURE_COLUMNS].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5,
            xticklabels=[f.capitalize() for f in FEATURE_COLUMNS],
            yticklabels=[f.capitalize() for f in FEATURE_COLUMNS])
plt.title('Matricea de Corelatie - Feature-uri Audio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature-uri pe Mood

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

mood_names = [MOOD_LABELS[i] for i in range(8)]

for i, feat in enumerate(FEATURE_COLUMNS):
    data = [df[df['mood'] == m][feat].values for m in mood_names]
    bp = axes[i].boxplot(data, labels=mood_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i].set_title(feat.capitalize(), fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Distributia Feature-urilor pe Mood', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/features_by_mood.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Profilul Audio Mediu pe Mood

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[FEATURE_COLUMNS]), columns=FEATURE_COLUMNS)
df_scaled['mood'] = df['mood'].values

fig, axes = plt.subplots(2, 4, figsize=(20, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()

categories = [f.capitalize() for f in FEATURE_COLUMNS]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for i, mood in enumerate(mood_names):
    vals = df_scaled[df_scaled['mood'] == mood][FEATURE_COLUMNS].mean().values.flatten().tolist()
    vals += vals[:1]
    
    axes[i].plot(angles, vals, 'o-', linewidth=2, color=colors[i])
    axes[i].fill(angles, vals, alpha=0.25, color=colors[i])
    axes[i].set_xticks(angles[:-1])
    axes[i].set_xticklabels(categories, size=7)
    axes[i].set_title(mood.capitalize(), fontsize=11, fontweight='bold', pad=15)
    axes[i].set_ylim(0, 1)

plt.suptitle('Profilul Audio Mediu pe Mood (Normalizat)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/mood_profiles_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Concluzii EDA

- Datasetul contine **10.000 de piese** cu **8 feature-uri audio** si **8 mood-uri**
- Exista un **dezechilibru de clasa** semnificativ (happy/energetic domina, romantic/calm sunt subreprezentate)
- `loudness` si `tempo` necesita normalizare (MinMaxScaler)
- Exista corelatii puternice intre energy-valence si energy-loudness
- Fiecare mood are un profil audio distinct, ceea ce favorizeaza clasificarea